In [16]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import pyarrow.feather as feather
import seaborn as sns
import seaborn.objects as so
from scipy.stats import kstest

import importlib
import sys
import os
hostname = os.uname().nodename
if hostname == 'BlackBeast':
    path = '/home/hedvigs/snake_book/econ'
    site = 'home'
elif hostname == 'hedvig-hp-elitedesk-800-g5-twr':
    path = '/home/hedvigs/PycharmProjects/homewrs/plab_workflow/'
    site = 'work'
elif hostname == 'work-computer':
    path = '/mnt/work/workbench/hedvigs/snake_book/econ'
    site = 'server'
elif hostname == 'wl-241113-007':
    path = '/home/hedvigs/wslGit/snake_book/econ'
    site = 'silverFlex'

sys.path.append(path)
print(path)
from workflow.scripts.data_management import setup_data as gt


/home/hedvigs/PycharmProjects/homewrs/plab_workflow/


In [17]:


def summarize_scores(subset=None, model=None, gen=None, fold=None, path=None, verbose=0):
    inputs = [subset, gen, fold, model]
    config_names = ['subsets', 'genomes', 'folds', 'models']

    config_dict = {}

    for item, config_name in zip(inputs, config_names):
        if item is not None:
            config_dict[config_name] = [item]
        else:
            config_dict[config_name] = gt.read_config(config_name,path=path)
    y_pred=[]
    missing = pd.DataFrame()
    k=0
    
    for subset in config_dict['subsets']:
        for model in config_dict['models']:
            for gen in config_dict['genomes']:
                for fold in config_dict['folds']:
                            score_file= path + f'results/scores/PTD/{subset}/{model}_{gen}_{fold}.csv'
                            try: 
                                scores = pd.read_csv(score_file, sep='\t')
                                scores['fold'] = fold
                                scores['gen'] = gen
                                scores['model'] = model
                                scores['subset'] = subset
                                y_pred.append(scores)
                            except FileNotFoundError:
                                k+=1
                                missing.loc[k, 'subset'] = subset
                                missing.loc[k, 'model'] = model
                                missing.loc[k, 'gen'] = gen
                                missing.loc[k, 'fold'] = fold
    if k == 0:
        print(f' All files found!, {k} missing')
    elif verbose == 1:
        print(f'Files not found:', k)
    elif verbose == 2:
        print(f'Files not found:', k)
        subsetnames, subsetcounts =np.unique(missing['subset'], return_counts=True)
        for sname, scount in zip(subsetnames, subsetcounts):
            print(f'  {sname}: {scount} missing')
            missubset = missing[missing['subset']==sname]
            modelnames, modelcounts =np.unique(missubset['model'], return_counts=True)
            for mname, mcount in zip(modelnames, modelcounts):
                print(f'    {mname}: {mcount} missing')
    elif verbose == 3:
        print('Total files not found:', k)
        for colname in missing.columns:
            names, counts = np.unique(missing[colname], return_counts=True)
            print(f'{colname}: ')
            for name, count in zip(names, counts):
                print(f'  {name}: {count} missing')
    elif verbose == 4:
        print(f'Files not found:', k)
        subsetnames, subsetcounts =np.unique(missing['subset'], return_counts=True)
        for sname, scount in zip(subsetnames, subsetcounts):
            print(f'  {sname}: {scount} missing')
            missubset = missing[missing['subset']==sname]
            modelnames, modelcounts =np.unique(missubset['model'], return_counts=True)
            for mname, mcount in zip(modelnames, modelcounts):
                print(f'    {mname}: {mcount} missing')
                missmodel = missubset[missubset['model']==mname]
                gennames, gencounts =np.unique(missmodel['gen'], return_counts=True)
                for gname, gcount in zip(gennames, gencounts):
                    print(f'        {gname}: {gcount} missing')

    full_set = pd.concat(y_pred)
    return full_set






In [18]:

    
subset = None #'top29' # wildcards["iSubset"]
model = None
gen = None  # wildcards["iGen"]

sum_df = summarize_scores(subset=subset, model=model, gen=gen, fold=None,path=path, verbose=4)
auc_df = sum_df.drop(columns=["number"]) # ,"fold", "train_score", "f1", "fbeta", "perm_score", "plr", "nlr", "pval_score", "auc_pred"])


Files not found: 317
  all: 120 missing
    bnb: 15 missing
        combine: 5 missing
        f: 5 missing
        m: 5 missing
    knn: 15 missing
        combine: 5 missing
        f: 5 missing
        m: 5 missing
    lda: 15 missing
        combine: 5 missing
        f: 5 missing
        m: 5 missing
    lrc: 15 missing
        combine: 5 missing
        f: 5 missing
        m: 5 missing
    nn: 15 missing
        combine: 5 missing
        f: 5 missing
        m: 5 missing
    qda: 15 missing
        combine: 5 missing
        f: 5 missing
        m: 5 missing
    rfc: 15 missing
        combine: 5 missing
        f: 5 missing
        m: 5 missing
    svc: 15 missing
        combine: 5 missing
        f: 5 missing
        m: 5 missing
  top29: 77 missing
    bnb: 10 missing
        combine: 1 missing
        f: 4 missing
        m: 5 missing
    knn: 10 missing
        combine: 5 missing
        f: 1 missing
        m: 4 missing
    lda: 10 missing
        combine: 4 missing
    

In [12]:
auc_df


,test_perm,test_pval,test_score,train_score,auc_prob,auc_pred,r2,tn,fp,fn,tp,fold,gen,model,subset
0,0.49853,0.03393,0.52770,0.59457,0.52770,0.50990,-9.20597,1424,1752,36,48,3,f,bnb,top29
1,0.50026,0.82435,0.48423,0.59447,0.48423,0.50478,-9.29357,1694,1482,44,40,3,f,bnb,top29
2,0.50079,0.12375,0.52078,0.60023,0.52078,0.50954,-9.27580,1573,1603,40,44,3,f,bnb,top29
3,0.50126,0.63473,0.49623,0.57944,0.49623,0.47585,-9.32600,1548,1628,45,39,3,f,bnb,top29
4,0.49835,0.33733,0.50406,0.60520,0.50406,0.51178,-9.57121,1625,1551,41,43,3,f,bnb,top29
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15,0.50024,0.00200,0.55064,0.77451,0.53725,0.50050,-8.95429,1629,1547,43,41,3,combine,svc,top29
16,0.50037,0.00200,0.55327,0.79482,0.55301,0.54847,-8.99119,1669,1507,36,48,3,combine,svc,top29
17,0.49973,0.00200,0.57500,0.79240,0.57499,0.56856,-8.84736,1532,1644,29,55,3,combine,svc,top29
18,0.50058,0.00399,0.55035,0.78088,0.55118,0.51471,-8.99376,1568,1608,39,45,3,combine,svc,top29


In [19]:

#sum_file= path + 'results/report/sum_file_top29.csv'
sum_file= path + f'results/report/sum_file_seltop.csv'
auc_df.to_csv(sum_file)
